This notebook is to visualize the figures and tables in the main document of the paper. 
We need to load the saved networks. For illustration purposes, we use the .pth files under the paper/artifacts/main_paper folder.

In [ ]:
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
for candidate in [PROJECT_ROOT, *PROJECT_ROOT.parents]:
    if (candidate / "network_generator").exists():
        PROJECT_ROOT = candidate
        break
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

import torch
import torch.nn as nn
from network_generator.config import NetworkConfig, TargetConfig, TrainingConfig
from network_generator.data_iot import (
    load_input_output_table,
)
from network_generator.diagnostics import compute_network_stats
from network_generator.visualization import visualize_network
from paper.scripts.diagnostics_latex_table import (
    diagnostics_to_latextable,
)
from paper.scripts.visualize_for_overleaf_figures import (
    visualize_network_for_overleaf_main,
    visualize_joint_distributions,
    visualize_iot,
    visualize_deg_ccdf,
)

criterion_sse = nn.MSELoss(reduction="sum")
criterion_mse = nn.MSELoss()

In [ ]:
# Configuration - modify these config classes to change parameters
network_config = NetworkConfig(num_firms=100_000)
target_config = TargetConfig()
training_config = TrainingConfig()

# For backward compatibility, extract values to individual variables
num_firms = network_config.num_firms
monetary_unit = network_config.monetary_unit
country_code = network_config.country_code
country_year = network_config.country_year

In [ ]:
# Extract target values from config for backward compatibility
target_hill_in_deg = torch.tensor(target_config.target_hill_in_deg, dtype=torch.float32)
target_hill_out_deg = torch.tensor(target_config.target_hill_out_deg, dtype=torch.float32)
target_hill_weight_distribution = torch.tensor(target_config.target_hill_weight_distribution, dtype=torch.float32)
target_hill_in_str = torch.tensor(target_config.target_hill_in_str, dtype=torch.float32)
target_hill_out_str = torch.tensor(target_config.target_hill_out_str, dtype=torch.float32)
target_average_connection = torch.tensor(target_config.target_average_connection, dtype=torch.float32)

target_so_ko_corr = torch.tensor(target_config.target_so_ko_corr, dtype=torch.float32)
target_si_ki_corr = torch.tensor(target_config.target_si_ki_corr, dtype=torch.float32)
target_si_so_corr = torch.tensor(target_config.target_si_so_corr, dtype=torch.float32)
target_ki_ko_corr = torch.tensor(target_config.target_ki_ko_corr, dtype=torch.float32)

target_so_ko_regcoeff = torch.tensor(target_config.target_so_ko_regcoeff, dtype=torch.float32)
target_ko_so_regcoeff = torch.tensor(target_config.target_ko_so_regcoeff, dtype=torch.float32)
target_si_ki_regcoeff = torch.tensor(target_config.target_si_ki_regcoeff, dtype=torch.float32)
target_ki_si_regcoeff = torch.tensor(target_config.target_ki_si_regcoeff, dtype=torch.float32)

target_tls_str = torch.tensor(target_config.target_tls_str, dtype=torch.float32)
target_tls_deg = torch.tensor(target_config.target_tls_deg, dtype=torch.float32)

In [ ]:
# Load a pre-saved model
wgt_mat_optim_init = torch.load("./paper/artifacts/main_paper/wgt_mat_optim_init.pth", weights_only=True)
in_deg_seq_burr = torch.load("./paper/artifacts/main_paper/in_deg_seq_burr.pth", weights_only=True)
out_deg_seq_burr = torch.load("./paper/artifacts/main_paper/out_deg_seq_burr.pth", weights_only=True)
test_reordered_out_deg_seq_burr = torch.load("./paper/artifacts/main_paper/test_reordered_out_deg_seq_burr.pth", weights_only=True)
adj_mat_init = torch.load("./paper/artifacts/main_paper/adj_mat_init.pth", weights_only=True)
in_deg_seq_init = torch.load("./paper/artifacts/main_paper/in_deg_seq_init.pth", weights_only=True)
out_deg_seq_init = torch.load("./paper/artifacts/main_paper/out_deg_seq_init.pth", weights_only=True)
in_str_seq_init = torch.load("./paper/artifacts/main_paper/in_str_seq_init.pth", weights_only=True)
out_str_seq_init = torch.load("./paper/artifacts/main_paper/out_str_seq_init.pth", weights_only=True)
firm_to_industry_tensor = torch.load("./paper/artifacts/main_paper/firm_to_industry_tensor.pth", weights_only=True)

data_iot_torch, data_iot_np, num_industries = load_input_output_table(
    data_path="data/",
    country_code=country_code,
    year=country_year,
    monetary_unit=monetary_unit,
)

In [ ]:
visualize_network(
    wgt_mat_optim_init,
    firm_to_industry_tensor,
    data_iot_torch,
    monetary_unit=monetary_unit,
    country_code=country_code,
    country_year=country_year,
)

In [ ]:
df_diagnostics_optim = compute_network_stats(
    model=wgt_mat_optim_init,
    firm_to_industry=firm_to_industry_tensor,
    data_iot=data_iot_torch,
    num_of_industries=num_industries,
    network_config=network_config,
    target_config=target_config,
)
df_diagnostics_optim

In [ ]:
print(diagnostics_to_latextable(
    df_diagnostics_optim,
    split_after_metric="IOT RMSE (100 MUSD)",
))

In [ ]:
# uncomment the second save_path line to save the network visualization into paper/artifacts/main_paper folder. 

visualize_network_for_overleaf_main(
    model=wgt_mat_optim_init,
    monetary_unit=monetary_unit,
    save_path = "./paper/artifacts/main_paper/network_visualization.png"
)

In [ ]:
# uncomment the second save_path line to save the network visualization into paper/artifacts/main_paper folder. 
visualize_iot(
    wgt_mat_optim_init,
    firm_to_industry_tensor,
    data_iot_np,
    num_industries,
    country_code,
    country_year,
    save_path = "./paper/artifacts/main_paper/iot_visualization.png",
)

In [ ]:
visualize_deg_ccdf(
    model = wgt_mat_optim_init,
    save_path = None
)

In [ ]:
visualize_joint_distributions(
    model=wgt_mat_optim_init, monetary_unit=monetary_unit, save_path=None
)